In [1]:
# ! pip install chromadb

In [2]:
! pip show chromadb

Name: chromadb
Version: 1.5.7
Summary: Chroma.
Home-page: https://github.com/chroma-core/chroma
Author: 
Author-email: Jeff Huber <jeff@trychroma.com>, Anton Troynikov <anton@trychroma.com>
License: 
Location: /Users/kanavbansal/Desktop/Hands-on Labs (Evoke)/evoke_b1/.env/lib/python3.13/site-packages
Requires: bcrypt, build, grpcio, httpx, importlib-resources, jsonschema, kubernetes, mmh3, numpy, onnxruntime, opentelemetry-api, opentelemetry-exporter-otlp-proto-grpc, opentelemetry-sdk, orjson, overrides, pybase64, pydantic, pydantic-settings, pypika, pyyaml, rich, tenacity, tokenizers, tqdm, typer, typing-extensions, uvicorn
Required-by: langchain-chroma


In [3]:
import chromadb

# Setup API Key
with open('keys/.chroma_api_key.txt') as f:
    CHROMA_API_KEY = f.read()

with open('keys/.chroma_tenant.txt') as f:
    CHROMA_TENANT = f.read()


client = chromadb.CloudClient(
  api_key=CHROMA_API_KEY,
  tenant=CHROMA_TENANT,
  database='chroma_db_1'
)

client.list_collections()

[]

In [4]:
# client.delete_collection("food-collection")

client.list_collections()

[]

## **Enabling Sparse Vector Index**

To use sparse vectors, add a sparse vector index to your schema. The **key** parameter is the metadata field name where sparse embeddings will be stored - you can name it whatever you want.

The **source_key** inside the SparseVectorIndexConfig specifies which field to generate sparse embeddings from (typically **K.DOCUMENT** for document text), and **embedding_function** specifies the function to generate the sparse embeddings. This example uses ChromaBm25EmbeddingFunction, but you can also use other sparse embedding functions like HuggingFaceSparseEmbeddingFunction or FastembedSparseEmbeddingFunction. The sparse embeddings are automatically generated and stored in the metadata field you specify as the key.

In [5]:
from chromadb import Schema, SparseVectorIndexConfig, K
from chromadb.utils.embedding_functions import ChromaBm25EmbeddingFunction

schema = Schema()

In [6]:
schema = schema.create_index(
  key='sparse_vector_key',                  # Mandatory to Pass - Name can be anything
  config=SparseVectorIndexConfig(
    source_key=K.DOCUMENT,
    bm25=True,
    embedding_function=ChromaBm25EmbeddingFunction(
        k=1.2,
        b=0.75,
        avg_doc_length=256.0,
        token_max_length=40
    ),
  )
)

In [7]:
# Custom Dense Vector Embedding (Semantic Embedding)

from chromadb import Schema, VectorIndexConfig
# from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction

# f = open("keys/.openai_api_key.txt")
# OPENAI_API_KEY = f.read()

schema = schema.create_index(
    config=VectorIndexConfig(
        source_key=K.DOCUMENT,
        space="cosine",
        # embedding_function=OpenAIEmbeddingFunction(
        #     api_key=OPENAI_API_KEY,
        #     model_name="text-embedding-3-small"
        # )
    )
)

In [8]:
client.list_collections()

[]

In [9]:
schema

Schema(defaults=ValueTypes(string=StringValueType(fts_index=FtsIndexType(enabled=False, config=FtsIndexConfig()), string_inverted_index=StringInvertedIndexType(enabled=True, config=StringInvertedIndexConfig())), float_list=FloatListValueType(vector_index=VectorIndexType(enabled=False, config=VectorIndexConfig(space='cosine', embedding_function=<chromadb.api.types.DefaultEmbeddingFunction object at 0x10aee16a0>, source_key='#document', hnsw=None, spann=None))), sparse_vector=SparseVectorValueType(sparse_vector_index=SparseVectorIndexType(enabled=False, config=SparseVectorIndexConfig(embedding_function=None, source_key=None, bm25=None))), int_value=IntValueType(int_inverted_index=IntInvertedIndexType(enabled=True, config=IntInvertedIndexConfig())), float_value=FloatValueType(float_inverted_index=FloatInvertedIndexType(enabled=True, config=FloatInvertedIndexConfig())), boolean=BoolValueType(bool_inverted_index=BoolInvertedIndexType(enabled=True, config=BoolInvertedIndexConfig()))), keys={

In [10]:
collection = client.get_or_create_collection(
    name="food-collection", 
    schema=schema,
)

In [11]:
client.list_collections()

[Collection(name=food-collection)]

In [12]:
collection.count()

0

In [13]:
collection.configuration

{'hnsw': None,
 'spann': {'search_nprobe': 64,
  'write_nprobe': 32,
  'space': 'cosine',
  'ef_construction': 200,
  'ef_search': 200,
  'max_neighbors': 64,
  'reassign_neighbor_count': 64,
  'split_threshold': 50,
  'merge_threshold': 25},
 'embedding_function': <chromadb.api.types.DefaultEmbeddingFunction at 0x110746e70>}

In [14]:
collection.peek()

{'ids': [],
 'embeddings': array([], dtype=float64),
 'metadatas': [],
 'documents': [],
 'data': None,
 'uris': None,
 'included': ['metadatas', 'documents', 'embeddings']}

In [15]:
data = [
    ("Apples - High in fiber, support digestion, and promote heart health.", "Fruit"),
    ("Bananas - Rich in potassium, help regulate blood pressure and muscle function.", "Fruit"),
    ("Oranges - Packed with vitamin C, boost immunity, and promote skin health.", "Fruit"),
    ("Blueberries - High in antioxidants, improve brain function and reduce inflammation.", "Fruit"),
    ("Strawberries - Support heart health and contain anti-aging antioxidants.", "Fruit"),
    ("Watermelon - Hydrating fruit with lycopene, good for heart and skin health.", "Fruit"),
    ("Pineapple - Contains bromelain, aids digestion, and reduces inflammation.", "Fruit"),
    ("Avocado - Loaded with healthy fats, supports brain and heart health.", "Fruit"),
    ("Papaya - Rich in enzymes for digestion and boosts skin health.", "Fruit"),
    ("Pomegranate - Full of antioxidants, improves blood circulation, and heart health.", "Fruit"),
    ("Carrots - High in beta-carotene, improve eye health and skin glow.", "Vegetable"),
    ("Spinach - Rich in iron, good for blood health and energy levels.", "Vegetable"),
    ("Broccoli - Contains sulforaphane, which has anti-cancer properties.", "Vegetable"),
    ("Tomatoes - Packed with lycopene, supports heart health and skin protection.", "Vegetable"),
    ("Bell Peppers - High in vitamin C, boosts immunity, and reduces inflammation.", "Vegetable"),
    ("Cucumber - Hydrating vegetable, aids in digestion, and supports skin health.", "Vegetable"),
    ("Garlic - Has antibacterial properties, supports heart health and immunity.", "Vegetable"),
    ("Ginger - Anti-inflammatory, helps with digestion and nausea relief.", "Vegetable"),
    ("Beets - Improve blood flow, support endurance, and detox the liver.", "Vegetable"),
    ("Sweet Potatoes - Rich in fiber and vitamin A, supports vision and digestion.", "Vegetable")
]

print(len(data))

20


In [17]:
import uuid

documents = [item[0] for item in data]
metadatas = [{"type": item[1]} for item in data]
ids = [str(uuid.uuid4()) for _ in data]

In [19]:
from openai import OpenAI

f = open("keys/.openai_api_key.txt")
OPENAI_API_KEY = f.read()

client = OpenAI(api_key=OPENAI_API_KEY)

response = client.embeddings.create(
    model="text-embedding-3-small",
    input=documents,
)

# Printing the first 50 elements from the embedding
print("First embd vector:", response.data[0].embedding[:10])
print()
print("Number of embeddings:", len(response.data))

First embd vector: [-0.012664794921875, -0.0160980224609375, -0.00299835205078125, 0.07647705078125, 0.004398345947265625, -0.0217437744140625, -0.013916015625, 0.01555633544921875, 0.03753662109375, 0.0056915283203125]

Number of embeddings: 20


In [20]:
embeddings = [ embd.embedding for embd in response.data ]

print("Number of embeddings:", len(embeddings))
print("Shape of each embedding:", len(embeddings[0]))

Number of embeddings: 20
Shape of each embedding: 1536


In [23]:
# ! pip install snowballstemmer

In [24]:
collection.add(
    documents=documents,
    embeddings=embeddings,
    ids=ids,
    metadatas=metadatas,
)


# Sparse embeddings for "sparse_vector_key" are generated automatically
# from the documents (source_key=K.DOCUMENT)

### **Sprase Vector Search**

#### **Search API, Knn and K**

The Search class accepts four optional parameters:
- where: Filter expressions to narrow down results
- rank: Ranking expressions to score and order results
- limit: Pagination control
- select: Fields to include in results - Available fields: #id, #document, #embedding, #metadata, #score, or any custom metadata field

**Base Search:**
- Filter search results using Where expressions and the Key class (aliased as K) to narrow down your search to specific documents, IDs, or metadata values.
```python
# Each method returns a new instance
base_search = Search().where(K("category") == "science")
```



In [26]:
from chromadb import Knn, K
from chromadb import Search

# Step 1: Search using sparse embeddings only
sparse_rank = Knn(
  query="full of fiber",  # Text query for sparse embeddings
  key="sparse_vector_key",  # Metadata field for sparse vectors
  return_rank=True,
  limit=5                 # Only the 5 nearest documents get scored (default limit 16) - HNSW uses this Limit
)

# Step 2: Build search
sparse_search = (Search()
  .rank(sparse_rank)
  .select(K.DOCUMENT, K.SCORE, K.METADATA)
  .limit(3)
)

# Step 3: Execute search
sparse_results = collection.search(sparse_search)

print(sparse_results)

{'ids': [['67b38cca-3349-4461-82ca-6b2bb758d11c', 'b526cb00-8d6e-4dbe-b810-9a086ca360ec', 'c50c5c02-f40b-42fe-b6ff-46437967d808']], 'documents': [['Apples - High in fiber, support digestion, and promote heart health.', 'Pomegranate - Full of antioxidants, improves blood circulation, and heart health.', 'Sweet Potatoes - Rich in fiber and vitamin A, supports vision and digestion.']], 'embeddings': [None], 'metadatas': [[{'type': 'Fruit', 'sparse_vector_key': SparseVector(indices=[64890871, 844567596, 1085451005, 1100855371, 1176022733, 1357846105, 1612531086, 1748360133], values=[1.6564705, 1.6564705, 1.6564705, 1.6564705, 1.6564705, 1.6564705, 1.6564705, 1.6564705], labels=None)}, {'sparse_vector_key': SparseVector(indices=[696572964, 844567596, 1058501323, 1357846105, 1648491894, 1768417799, 1912070388, 2012469550], values=[1.6564705, 1.6564705, 1.6564705, 1.6564705, 1.6564705, 1.6564705, 1.6564705, 1.6564705], labels=None), 'type': 'Fruit'}, {'type': 'Vegetable', 'sparse_vector_key':

In [28]:
# Let's perform the search again, but this time with metadata filtering

from chromadb import Knn, K
from chromadb import Search

# Step 1: Search using sparse embeddings only
sparse_rank = Knn(
  query="full of fiber",  # Text query for sparse embeddings
  key="sparse_vector_key",  # Metadata field for sparse vectors
  return_rank=True,
  limit=5                 # Only the 5 nearest documents get scored (default limit 16) - HNSW uses this Limit
)

# Step 2: Build search
sparse_search = (Search()
  .rank(sparse_rank)
  .select(K.DOCUMENT, K.SCORE, K.METADATA)
  .where(K("type")=="Fruit")
  .limit(3)
)

# Step 3: Execute search
sparse_results = collection.search(sparse_search)

print(sparse_results)

{'ids': [['67b38cca-3349-4461-82ca-6b2bb758d11c', 'b526cb00-8d6e-4dbe-b810-9a086ca360ec', 'c70dbbc8-05eb-49fc-80f1-f8906e3f7f1f']], 'documents': [['Apples - High in fiber, support digestion, and promote heart health.', 'Pomegranate - Full of antioxidants, improves blood circulation, and heart health.', 'Bananas - Rich in potassium, help regulate blood pressure and muscle function.']], 'embeddings': [None], 'metadatas': [[{'type': 'Fruit', 'sparse_vector_key': SparseVector(indices=[64890871, 844567596, 1085451005, 1100855371, 1176022733, 1357846105, 1612531086, 1748360133], values=[1.6564705, 1.6564705, 1.6564705, 1.6564705, 1.6564705, 1.6564705, 1.6564705, 1.6564705], labels=None)}, {'type': 'Fruit', 'sparse_vector_key': SparseVector(indices=[696572964, 844567596, 1058501323, 1357846105, 1648491894, 1768417799, 1912070388, 2012469550], values=[1.6564705, 1.6564705, 1.6564705, 1.6564705, 1.6564705, 1.6564705, 1.6564705, 1.6564705], labels=None)}, {'type': 'Fruit', 'sparse_vector_key': S

In [32]:
for row in sparse_results.rows()[0]:
    print(f"Score: {row['score']:.3f} - {row['document']} - {row['metadata']['type']}")
    print()

Score: 0.000 - Apples - High in fiber, support digestion, and promote heart health. - Fruit

Score: 1.000 - Pomegranate - Full of antioxidants, improves blood circulation, and heart health. - Fruit

Score: 2.000 - Bananas - Rich in potassium, help regulate blood pressure and muscle function. - Fruit



### **Dense Vector Search**

In [35]:
from chromadb import Knn
from chromadb import Search

# Dense semantic embeddings
dense_rank = Knn(
  query="full of fiber",  # Text query for dense embeddings
  key="#embedding",          # Default embedding field
  return_rank=True,
)

# Build and execute search
dense_search = (Search()
  .rank(dense_rank)
  .select(K.DOCUMENT, K.SCORE, K.METADATA)   # You can add K.EMBEDDING, K.ID
  .where(K("type")=="Vegetable")
  .limit(4)
)
  
dense_results = collection.search(dense_search)

print(dense_results)

{'ids': [['ac0b7e20-e0b3-4651-a6f5-843fe936edb8', '1f0a96aa-5788-474f-b97b-079f6acaf554', 'aec6beff-6053-4ea3-9dba-34c16235b4a9', 'c50c5c02-f40b-42fe-b6ff-46437967d808']], 'documents': [['Spinach - Rich in iron, good for blood health and energy levels.', 'Bell Peppers - High in vitamin C, boosts immunity, and reduces inflammation.', 'Ginger - Anti-inflammatory, helps with digestion and nausea relief.', 'Sweet Potatoes - Rich in fiber and vitamin A, supports vision and digestion.']], 'embeddings': [None], 'metadatas': [[{'sparse_vector_key': SparseVector(indices=[115441729, 380541707, 787810591, 1357846105, 1437406974, 1742567733, 1768417799, 1914358190], values=[1.6564705, 1.6564705, 1.6564705, 1.6564705, 1.6564705, 1.6564705, 1.6564705, 1.6564705], labels=None), 'type': 'Vegetable'}, {'type': 'Vegetable', 'sparse_vector_key': SparseVector(indices=[155292733, 516762017, 675648184, 1125453957, 1253056104, 1437973278, 1515533186, 1612531086, 1778635636], values=[1.6520973, 1.6520973, 1.6

In [30]:
for row in dense_results.rows()[0]:
    print(f"Embedding Length: {len(row['embedding'])}")
    print(f"Score: {row['score']:.3f} - {row['document']} - {row['metadata']['type']}")
    print()

Embedding Length: 1536
Score: 0.000 - Spinach - Rich in iron, good for blood health and energy levels. - Vegetable

Embedding Length: 1536
Score: 1.000 - Bell Peppers - High in vitamin C, boosts immunity, and reduces inflammation. - Vegetable

Embedding Length: 1536
Score: 2.000 - Ginger - Anti-inflammatory, helps with digestion and nausea relief. - Vegetable

Embedding Length: 1536
Score: 3.000 - Sweet Potatoes - Rich in fiber and vitamin A, supports vision and digestion. - Vegetable



### **Hybrid Search**

#### **RRF is a convinience wrapper**
Rrf is a convenience class that constructs the underlying ranking expression. You can manually build the same expression if needed:
```python
manual_rrf = -0.7 / (60 + dense_rank) - 0.3 / (60 + sparse_rank)
```

In [33]:
from chromadb import Rrf

# Combine with RRF
hybrid_rank = Rrf(
  ranks=[dense_rank, sparse_rank],
  weights=[0.7, 0.3],  # 70% semantic, 30% keyword
  k=60                 # smoothing parameter - higher values reduce emphasis on top ranks
)

# Use in search
hybrid_search = (Search()
  .rank(hybrid_rank)
  .limit(3)
  .select(K.DOCUMENT, K.SCORE, K.METADATA)
)

hybrid_results = collection.search(hybrid_search)

print(hybrid_results)

{'ids': [['c70dbbc8-05eb-49fc-80f1-f8906e3f7f1f', 'c50c5c02-f40b-42fe-b6ff-46437967d808', '93b9b33d-fba4-47d4-836d-3ea81f4a8049']], 'documents': [['Bananas - Rich in potassium, help regulate blood pressure and muscle function.', 'Sweet Potatoes - Rich in fiber and vitamin A, supports vision and digestion.', 'Oranges - Packed with vitamin C, boost immunity, and promote skin health.']], 'embeddings': [None], 'metadatas': [[{'type': 'Fruit', 'sparse_vector_key': SparseVector(indices=[178115665, 522609393, 874549584, 1031134330, 1137493292, 1402904868, 1768417799, 1895683639, 1914358190], values=[1.6520973, 1.6520973, 1.6520973, 1.6520973, 1.6520973, 1.6520973, 1.6520973, 1.6520973, 1.6520973], labels=None)}, {'sparse_vector_key': SparseVector(indices=[64890871, 446669981, 738409095, 1100855371, 1176022733, 1515533186, 1914358190, 2088618400], values=[1.6564705, 1.6564705, 1.6564705, 1.6564705, 1.6564705, 1.6564705, 1.6564705, 1.6564705], labels=None), 'type': 'Vegetable'}, {'sparse_vector

In [34]:
# Process results
for row in hybrid_results.rows()[0]:
    print(f"Score: {row['score']:.3f} - {row['document']} - {row['metadata']['type']}")

Score: -0.016 - Bananas - Rich in potassium, help regulate blood pressure and muscle function. - Fruit
Score: -0.015 - Sweet Potatoes - Rich in fiber and vitamin A, supports vision and digestion. - Vegetable
Score: -0.015 - Oranges - Packed with vitamin C, boost immunity, and promote skin health. - Fruit


In [36]:
hybrid_results.rows()[0]

[{'id': 'c70dbbc8-05eb-49fc-80f1-f8906e3f7f1f',
  'document': 'Bananas - Rich in potassium, help regulate blood pressure and muscle function.',
  'metadata': {'type': 'Fruit',
   'sparse_vector_key': SparseVector(indices=[178115665, 522609393, 874549584, 1031134330, 1137493292, 1402904868, 1768417799, 1895683639, 1914358190], values=[1.6520973, 1.6520973, 1.6520973, 1.6520973, 1.6520973, 1.6520973, 1.6520973, 1.6520973, 1.6520973], labels=None)},
  'score': -0.016052227},
 {'id': 'c50c5c02-f40b-42fe-b6ff-46437967d808',
  'document': 'Sweet Potatoes - Rich in fiber and vitamin A, supports vision and digestion.',
  'metadata': {'sparse_vector_key': SparseVector(indices=[64890871, 446669981, 738409095, 1100855371, 1176022733, 1515533186, 1914358190, 2088618400], values=[1.6564705, 1.6564705, 1.6564705, 1.6564705, 1.6564705, 1.6564705, 1.6564705, 1.6564705], labels=None),
   'type': 'Vegetable'},
  'score': -0.015132827},
 {'id': '93b9b33d-fba4-47d4-836d-3ea81f4a8049',
  'document': 'Orang